# Profil belge — résultats du backtest

Backtest du profil retenu (`config/profil_belge.yaml`) : 8 ETF UCITS irlandais de capitalisation, portefeuille **vu en EUR**,
parité de risque par classe d'actifs, vol cible 12 %, levier jusqu'à 2×, filtre de tendance lent, limite de perte de −15 %
(CPPI, réduction forcée, marge de chute, plus haut érodé), impôts belges compris (TOB, Reynders à la revente, taxe de 10 %
sur les plus-values appliquée à tout l'historique).

Comparé à des références calculées sur **les mêmes séries et avec les mêmes impôts** : 60/40, All-Weather, risk parity mensuelle,
actions mondiales, buy & hold.

Ce notebook ne télécharge rien : il lit le rapport JSON produit par

```bash
python -m trading_engine.backtest --overlay config/backtest_daily.yaml --overlay config/profil_belge.yaml \
  run --data data/backtest/daily_eur.jsonl --grid allocation.gross_band=0.10 --variants profil --series \
  --out data/backtest/report_notebook.json
```

Le workflow GitHub `notebook` fait tourner ce backtest sur les données réelles (Yahoo, BCE) et pousse le notebook exécuté.
**Pas un conseil en investissement ni un conseil fiscal.**

In [ ]:
import json, os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORT = Path(os.environ.get("REPORT", ROOT / "data/backtest/report_notebook.json"))
report = json.loads(REPORT.read_text(encoding="utf-8"))
if not report.get("series"):
    raise SystemExit("rapport sans séries : relancer le backtest avec --series")

# Palette catégorielle validée (ordre fixe, une couleur par stratégie, jamais recyclée).
COLORS = {"Profil belge": "#2a78d6", "60/40": "#eb6834", "All-Weather": "#1baf7a", "Risk parity mensuelle": "#eda100"}
MUTED, INK, INK2, GRID, SURFACE = "#9a9893", "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
plt.rcParams.update({
    "figure.figsize": (11, 4.6), "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlecolor": INK, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK2, "ytick.color": INK2,
    "lines.linewidth": 2.0, "legend.frameon": False, "font.size": 10,
})

def short(name):
    for key, label in (("Profil belge", "Profil belge"), ("60/40", "60/40"), ("All-Weather", "All-Weather"),
                       ("Risk parity", "Risk parity mensuelle"), ("Actions mondiales", "Actions mondiales"),
                       ("Buy & hold", "Buy & hold")):
        if key in name:
            return label + (" (walk-forward)" if "walk-forward" in name else "")
    return name

rows = {short(r["name"]): r for r in report["rows"] if "walk-forward" not in r["name"]}
equity = {short(n): pd.Series({pd.Timestamp(d): v for d, v in pts}) for n, pts in report["series"]["equity_eur"].items()
          if "walk-forward" not in n}
exposure = {short(n): pd.Series({pd.Timestamp(d): v for d, v in pts}) for n, pts in report["series"]["exposure"].items()}
MAIN = ["Profil belge", "60/40", "All-Weather", "Risk parity mensuelle"]
print(f"Évaluation {report['eval_start']} → {report['eval_end']} ; capital initial "
      f"{report['initial_cash']:,.0f} {report['currency']} ; {len(rows)} stratégies")

## Résumé

Rendement annualisé en EUR, avant et après liquidation (tout vendu à la fin : TOB, Reynders, plus-values, frais), risque et coûts.

In [ ]:
def pct(x):
    return None if x is None else round(100 * x, 1)

table = pd.DataFrame({
    name: {
        "CAGR EUR %": pct(r["metrics"].get("cagr_eur")),
        "CAGR liquidé %": pct(r["metrics"].get("cagr_eur_net")),
        "Volatilité %": pct(r["metrics"].get("vol_eur")),
        "Sharpe": None if r["metrics"].get("sharpe_eur") is None else round(r["metrics"]["sharpe_eur"], 2),
        "Sharpe IC 90 %": (f"{r['sharpe_ci']['low']:.2f} – {r['sharpe_ci']['high']:.2f}"
                           if (r.get("sharpe_ci") or {}).get("low") is not None else "–"),
        "Perte max EUR %": pct(r["metrics"].get("max_drawdown_eur")),
        "Rotation / an": None if r["metrics"].get("turnover") is None else round(r["metrics"]["turnover"], 2),
        "TOB EUR": round(r["metrics"].get("transaction_tax") or 0),
        "Impôt PV EUR": round((r.get("stats") or {}).get("gains_tax_paid") or 0),
        "Reynders EUR": round((r.get("stats") or {}).get("interest_tax_paid") or 0),
    } for name, r in rows.items()
}).T
table

## Valeur du portefeuille (EUR, échelle logarithmique)

In [ ]:
fig, ax = plt.subplots()
for name in MAIN:
    s = equity[name] / equity[name].iloc[0] * 100
    ax.plot(s.index, s.values, color=COLORS[name], label=name)
    ax.annotate(f"{name}  {s.iloc[-1]:.0f}", (s.index[-1], s.iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color=INK, fontsize=9)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:.0f}"))
ax.yaxis.set_minor_formatter(mtick.NullFormatter())
ax.yaxis.set_major_locator(mtick.LogLocator(base=10, subs=(1, 1.25, 1.5, 2, 2.5, 3, 4, 5, 6, 8)))
ax.set_title("Valeur de 100 € investis, impôts compris")
ax.set_ylabel("base 100 (log)")
ax.legend(loc="upper left")
ax.margins(x=0.12)
plt.show()

## Baisses depuis le plus haut (EUR)

La ligne pointillée est la limite de perte du profil (−15 %).

In [ ]:
fig, ax = plt.subplots()
for name in ["Profil belge", "60/40", "All-Weather"]:
    s = equity[name]
    dd = (s / s.cummax() - 1) * 100
    ax.plot(dd.index, dd.values, color=COLORS[name], label=f"{name} (pire {dd.min():.1f} %)")
ax.axhline(-15, color=INK2, linewidth=1, linestyle=(0, (4, 3)))
ax.text(1.005, -15, "−15 %", transform=ax.get_yaxis_transform(), ha="left", va="center",
        color=INK2, fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("Baisse depuis le plus haut")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3)
plt.show()

worst = pd.DataFrame({
    name: {"Baisse %": pct(w["depth"]), "Sommet": w["peak"], "Creux": w["trough"], "Retour au sommet": w["recovered"] or "pas encore"}
    for name, r in rows.items() if (w := r["metrics"].get("worst_drawdown_eur"))
}).T
worst

## Rendement par année

In [ ]:
names = ["Profil belge", "60/40", "All-Weather"]
years = sorted({y for n in names for y in rows[n].get("years", {})})
yearly = pd.DataFrame({n: [100 * (rows[n]["years"].get(y) or 0) for y in years] for n in names}, index=years)
fig, ax = plt.subplots(figsize=(11, 4.8))
width = 0.27
for i, n in enumerate(names):
    ax.bar([k + (i - 1) * (width + 0.02) for k in range(len(years))], yearly[n], width=width,
           color=COLORS[n], label=n, edgecolor=SURFACE, linewidth=1)
ax.axhline(0, color=INK2, linewidth=0.8)
ax.set_xticks(range(len(years)), [str(y) for y in years], rotation=45)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("Rendement annuel en EUR")
ax.legend(loc="upper left", ncol=3)
plt.show()
yearly.round(1).T

## Pendant les crises

Rendement de chaque stratégie sur la période de la crise.

In [ ]:
crises = [c["name"] for c in report.get("crises", [])]
crisis = pd.DataFrame({n: [100 * ((rows[n].get("crises") or {}).get(c) or {}).get("return", float("nan")) for c in crises]
                       for n in names}, index=crises)
fig, ax = plt.subplots(figsize=(11, 0.55 * len(crises) + 1.2))
height = 0.26
for i, n in enumerate(names):
    ax.barh([k + (i - 1) * (height + 0.02) for k in range(len(crises))], crisis[n], height=height,
            color=COLORS[n], label=n, edgecolor=SURFACE, linewidth=1)
ax.axvline(0, color=INK2, linewidth=0.8)
ax.set_yticks(range(len(crises)), crises)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("Rendement pendant chaque crise (EUR)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
ax.grid(axis="y", visible=False)
plt.show()
crisis.round(1)

## Exposition du profil

Exposition brute (somme des positions / valeur du portefeuille). Au-dessus de 1, le portefeuille emprunte ; le plafond est 2×. Elle baisse quand la tendance se retourne ou quand une perte approche la limite.

In [ ]:
expo = exposure[next(n for n in exposure if n.startswith("Profil belge"))]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(expo.index, expo.values, color=COLORS["Profil belge"], linewidth=1.5)
ax.axhline(1, color=INK2, linewidth=0.8, linestyle=(0, (4, 3)))
ax.axhline(2, color=INK2, linewidth=0.8, linestyle=(0, (1, 2)))
ax.annotate("sans levier", (expo.index[0], 1), xytext=(0, 4), textcoords="offset points", color=INK2, fontsize=9)
ax.annotate("plafond 2×", (expo.index[0], 2), xytext=(0, -12), textcoords="offset points", color=INK2, fontsize=9)
ax.set_ylim(0, 2.2)
ax.set_title("Exposition brute du profil belge")
plt.show()
print(f"Exposition moyenne {expo.mean():.2f} ; au-dessus de 1 {100 * (expo > 1).mean():.0f} % du temps ; "
      f"sous 0,5 {100 * (expo < 0.5).mean():.0f} % du temps")

## Rendement contre risque

Volatilité annuelle (EUR) et rendement après liquidation ; la taille du point suit la pire baisse.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2))
for name, r in rows.items():
    m = r["metrics"]
    x, y = 100 * (m.get("vol_eur") or 0), 100 * (m.get("cagr_eur_net") or m.get("cagr_eur") or 0)
    size = 40 + 12 * abs(100 * (m.get("max_drawdown_eur") or 0))
    color = COLORS["Profil belge"] if name == "Profil belge" else MUTED
    ax.scatter([x], [y], s=size, color=color, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.annotate(f"{name} ({100 * (m.get('max_drawdown_eur') or 0):.0f} %)", (x, y), xytext=(8, 4),
                textcoords="offset points", color=INK, fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.set_xlabel("volatilité annuelle")
ax.set_ylabel("rendement annuel après liquidation")
ax.set_title("Rendement contre risque (entre parenthèses : pire baisse)")
ax.margins(0.25)
plt.show()

## Le profil bat-il les références ?

Écart de Sharpe (profil − référence), intervalle de confiance à 90 % par bootstrap par blocs de 20 séances, et probabilité que l'écart soit positif. Sur 20 ans, une probabilité de 80-90 % n'est pas une preuve.

In [ ]:
vs = rows["Profil belge"].get("vs") or {}
pd.DataFrame({short(b): {"écart de Sharpe": round(d["difference"], 2) if d["difference"] is not None else None,
                          "IC 90 %": f"{d['low']:+.2f} à {d['high']:+.2f}" if d.get("low") is not None else "–",
                          "P(écart > 0) %": None if d.get("p_better") is None else round(100 * d["p_better"])}
              for b, d in vs.items()}).T

## Limites et traçabilité

In [ ]:
for c in report.get("caveats", []):
    print("-", c)
p = report.get("provenance") or {}
print()
print("données :", p.get("dataset_sha256", "")[:16], "| code :", p.get("git_commit"), "| généré :", p.get("created_at"))